In [1]:
import sys
print(sys.executable) 

/Users/jatin/Developer/Ml Project/college-admission-mlflow/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/College_Admission.csv')
df.head()

,student_id,age,gender,category,state,preferred_stream,entrance_exam,entrance_score,board_percentage,extracurricular_score,admission_probability,admission_status,scholarship_eligibility
0,250.99,17,other,general,odisha,management,cet,30,95.58,2,0.387,admitted,yes
1,250.99,20,other,ews,gujarat,agriculture,none,0,75.45,2,0.221,rejected,no
2,250.99,19,female,sc,uttar pradesh,pharmacy,cet,120,75.36,10,0.446,rejected,no
3,250.99,18,male,ews,meghalaya,arts,cet,179,52.49,2,0.174,admitted,no
4,250.99,18,male,sc,rajasthan,engineering,jee,295,92.48,7,0.634,admitted,yes


In [3]:
df.shape

(25000, 13)

In [4]:
df.describe()

,student_id,age,entrance_score,board_percentage,extracurricular_score,admission_probability
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,12500.500000,18.492440,77.879760,75.028014,4.990920,0.323997
std,7212.732314,1.116205,120.356125,14.438767,3.158942,0.142427
min,250.990000,17.000000,0.000000,50.440000,0.000000,0.041990
25%,6250.750000,17.000000,0.000000,62.550000,2.000000,0.219000
50%,12500.500000,18.000000,20.000000,75.070000,5.000000,0.319000
75%,18750.250000,19.000000,126.000000,87.640000,8.000000,0.419000
max,24750.010000,20.000000,634.000000,99.530000,10.000000,0.731010


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   student_id               25000 non-null  float64
 1   age                      25000 non-null  int64  
 2   gender                   25000 non-null  object 
 3   category                 25000 non-null  object 
 4   state                    25000 non-null  object 
 5   preferred_stream         25000 non-null  object 
 6   entrance_exam            25000 non-null  object 
 7   entrance_score           25000 non-null  int64  
 8   board_percentage         25000 non-null  float64
 9   extracurricular_score    25000 non-null  int64  
 10  admission_probability    25000 non-null  float64
 11  admission_status         25000 non-null  object 
 12  scholarship_eligibility  25000 non-null  object 
dtypes: float64(3), int64(3), object(7)
memory usage: 2.5+ MB


In [6]:
df.isna().sum()

student_id                 0
age                        0
gender                     0
category                   0
state                      0
preferred_stream           0
entrance_exam              0
entrance_score             0
board_percentage           0
extracurricular_score      0
admission_probability      0
admission_status           0
scholarship_eligibility    0
dtype: int64

In [7]:
df['admission_status'].value_counts(normalize=True)

admission_status
rejected    0.67392
admitted    0.32608
Name: proportion, dtype: float64

The data is imbalanced: 67.4% of students were rejected and only 32.6% were admitted.

This matters for how I measure the model. If a model simply predicted "rejected" for every single student, it would be correct 67.4% of the time — without looking at any of the student's features. So a high accuracy score does not prove the model is doing anything useful.

Because of this, I use ROC-AUC as my main metric instead of accuracy.

The model gives each student a score for how likely they are to be admitted. ROC-AUC measures whether the model gives higher scores to students who were actually admitted. An AUC of 1.0 means the ranking is perfect, and 0.5 means the model is no better than random guessing.

A "predict rejected for everyone" model gives every student the same score, so it ranks nothing and its AUC is 0.5 — which correctly shows it is useless, even though its accuracy looks acceptable.

I still report accuracy, precision, recall and F1 for completeness, but AUC is the metric I use to choose between models.

In [11]:
pd.crosstab(df['scholarship_eligibility'], df['admission_status'], normalize='index')

admission_status,admitted,rejected
scholarship_eligibility,,
no,0.155319,0.844681
yes,1.000000,0.000000


In [12]:
df.groupby('admission_status')['admission_probability'].describe()

,count,mean,std,min,25%,50%,75%,max
admission_status,,,,,,,,
admitted,8152.0,0.386653,0.138194,0.04199,0.288,0.382,0.471,0.73101
rejected,16848.0,0.293681,0.134322,0.04199,0.194,0.288,0.384,0.73101


In [13]:
from sklearn.metrics import roc_auc_score
y = (df['admission_status'] == 'admitted').astype(int)
roc_auc_score(y, df['admission_probability'])

0.6847012429205961

## Data leakage check

Two columns in this dataset give away the answer, so both are removed before training.

### `scholarship_eligibility`

Every student marked "yes" was admitted — 100%, with no exceptions across 25,000 rows.
Students marked "no" were admitted only 15.5% of the time.

This is not a predictive signal, it is the answer stored in a different column. A college
decides to admit a student first, and only then decides whether that student receives a
scholarship. For a real applicant whose decision has not yet been made, this column would
be empty. Training on it would teach the model a single rule — "yes means admitted" — that
cannot be applied at prediction time.

### `admission_probability`

This is the score used to generate the label, so it would not exist for a real applicant
either. It is a weaker giveaway than the scholarship column: admitted students average
0.387 and rejected students average 0.294, and the two ranges overlap heavily
(both span 0.042 to 0.731).

### Implication for expected performance

Using `admission_probability` directly as a prediction gives **0.685 ROC-AUC**. Since this
is the column the label was generated from, that value is approximately the ceiling for
any model on this dataset.

It is well below 1.0 because the column does not determine the outcome — it sets the odds.
A student with a probability of 0.6 had a 60% chance of admission, not a guarantee. Two
students with identical features can therefore receive different outcomes, meaning part of
the label is irreducibly random and no model can predict it.

A realistic target for this project is therefore **0.68–0.70 ROC-AUC**. An honest 70% built
on features available at prediction time is more valuable than a 100% built on leaked
columns.

### Columns dropped

| Column | Reason |
|---|---|
| `scholarship_eligibility` | Target leakage — determined after admission |
| `admission_probability` | Target leakage — the label's generating score |
| `student_id` | Identifier with no predictive meaning |